# 101 · Data science perspective lab

This notebook goes with the article
[Data science perspective](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/101/data-science-perspective/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/101/data_science_perspective.ipynb)

You will work with one small table of sample rows and try several common ways of storing it:
comma-separated values (CSV), JSON Lines, Python’s `pickle`, and a simple row-versus-column layout in memory.
Everything uses the Python standard library unless you optionally install `pyarrow` for the last exercise.

> **A note on numbers:** sizes and timings in these notebooks are only illustrations. For measured library comparisons on this project’s harness, use the suite [Results](https://leo-gan.github.io/GLD.SerializerBenchmark/) pages.


In [ ]:
import csv
import io
import json
import pickle
import struct
from pathlib import Path

# Tiny "lake" sample — wide enough to show column projection idea
ROWS = [
    {"event_id": i, "user": f"u{i%5}", "feature_a": i * 0.1, "feature_b": i % 3,
     "label": "pos" if i % 4 == 0 else "neg", "payload_note": "x" * 8}
    for i in range(100)
]
print(len(ROWS), "rows; columns:", list(ROWS[0]))



## CSV as a lossy social format

CSV is the format people reach for when they want something a spreadsheet can open.
In this cell you write the sample table to CSV and read it back.

Notice that every value returns as a **string**, even fields that started as numbers.
That is fine for a quick export to a human, but it is a weak choice as the long-term “source of truth”
for a data lake or machine-learning pipeline where types and missing values must stay reliable.


In [ ]:
buf = io.StringIO()
w = csv.DictWriter(buf, fieldnames=list(ROWS[0]))
w.writeheader()
w.writerows(ROWS)
csv_text = buf.getvalue()
print("CSV nbytes:", len(csv_text.encode()))
print(csv_text.splitlines()[0])
print(csv_text.splitlines()[1])
# Types are gone — everything is text on reload
buf.seek(0)
reread = list(csv.DictReader(buf))
print("feature_a type after CSV round-trip:", type(reread[0]["feature_a"]), reread[0]["feature_a"])



## JSON Lines as a landing zone

JSON Lines (sometimes written JSONL) stores **one JSON object per line**.
That makes files easy to append and easy to split for parallel processing.

When you re-read a line, ordinary JSON types come back (for example a number stays a number).
The cost is size: field names are repeated on every line, so the file is usually larger than CSV.
Treat JSON Lines as a good **landing** or log format, not as a substitute for columnar storage
when you need to scan a few columns over millions of rows.


In [ ]:
jsonl = "\n".join(json.dumps(r, separators=(",", ":")) for r in ROWS).encode()
print("JSONL nbytes:", len(jsonl))
first = json.loads(jsonl.splitlines()[0])
print("feature_a type after JSONL:", type(first["feature_a"]), first["feature_a"])



## Pickle and the trust boundary

Python’s `pickle` can save almost any in-memory object and restore it later.
That convenience is why it shows up in notebooks and some machine-learning checkpoints.

Run a round-trip only on data **you** just created—that is a trusted path.
The important rule for real systems is different: if the bytes might come from someone else,
from another team’s bucket, or from the network, **do not unpickle them**.
Loading untrusted pickle data can execute code. For sharing across languages or untrusted sources,
prefer portable formats such as JSON Lines, Parquet, or Arrow.


In [ ]:
trusted = pickle.dumps(ROWS)
print("pickle nbytes:", len(trusted))
assert pickle.loads(trusted)[0]["event_id"] == 0

# Illustrative hostile pattern (do NOT run on untrusted bytes in production).
# pickle can invoke callables during load — we only show the *opcode surface*, not a live exploit.
print("pickle protocol opcodes (prefix):", trusted[:20])
print("OK: portable formats (JSONL/Parquet/Arrow) for multi-language or untrusted interchange")



## Row layout versus column layout (toy model)

Analytics often needs only one field across many rows (for example, sum a single feature).
Here the same data is stored two ways: a list of dictionaries (row-oriented) and a dictionary of lists (column-oriented).

You should see the column-oriented path finish faster for that single-field sum.
Real engines such as Parquet and Arrow take the same idea further: they can skip entire columns on disk
instead of parsing whole records. That is why “we already log events as JSON” is a weak design for a large analytic lake.


In [ ]:
# Row store: list of dicts (pointer-rich)
row_store = ROWS

# Column store: one list per column
col_store = {k: [r[k] for r in ROWS] for k in ROWS[0]}


def sum_feature_a_rows():
    return sum(r["feature_a"] for r in row_store)


def sum_feature_a_cols():
    return sum(col_store["feature_a"])


import timeit
from statistics import median

t_row = median(timeit.repeat(sum_feature_a_rows, number=2000, repeat=5))
t_col = median(timeit.repeat(sum_feature_a_cols, number=2000, repeat=5))
assert abs(sum_feature_a_rows() - sum_feature_a_cols()) < 1e-9
print(f"sum feature_a via rows: {t_row*1e3:.3f} ms median")
print(f"sum feature_a via cols: {t_col*1e3:.3f} ms median")
print("Columnar wins more as width grows and engines skip unread columns (Parquet/Arrow idea).")



## Optional: Parquet and Arrow (if `pyarrow` is installed)

If the `pyarrow` package is available, this cell writes a small Parquet file and reads back only the `feature_a` column.
If the package is missing, the cell prints SKIP—that is expected on a minimal install.

The pattern to remember is simple: **Parquet** is a common on-disk columnar format for lakes;
**Arrow** is a common in-memory columnar layout when engines hand tables to each other.
You do not need both libraries for every job, but you will meet both names often in data platforms.


In [ ]:
try:
    import pyarrow as pa
    import pyarrow.parquet as pq

    table = pa.Table.from_pylist(ROWS)
    sink = pa.BufferOutputStream()
    pq.write_table(table, sink, compression="zstd")
    parquet_bytes = sink.getvalue().to_pybytes()
    print("Parquet nbytes:", len(parquet_bytes))
    # Project one column only
    col_only = pq.read_table(pa.BufferReader(parquet_bytes), columns=["feature_a"])
    print("read feature_a only:", col_only.column(0).to_pylist()[:5], "…")
except ImportError:
    print("SKIP pyarrow — optional: pip install pyarrow")
    print("Pattern to remember: Parquet on disk / Arrow in memory between engines.")



## Takeaways

| Kind of work | A sensible default |
|--------------|--------------------|
| Human-edited export | CSV (at the edge only) |
| Landing zone or line-oriented logs | JSON Lines |
| Large analytic scans | Columnar formats (Parquet-class) |
| Hand-off between data engines | Arrow when your tools support it |
| Untrusted or multi-language interchange | Never use pickle |

**Next:** try the [Engineering mini lab](./engineering_perspective.ipynb), or continue reading
[Serialization 201](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/201/).
